In [ ]:
# ========== 导入：环境、OpenAI、LangChain RAG、可视化、评估 ==========

# 标准库 os：读环境变量（如 OPENAI_API_KEY）
import os
# 标准库 glob：按模式找文件（本笔记本其它格也可能用到）
import glob
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# Path：拼路径（本格随其它依赖一并导入）
from pathlib import Path
# gradio：最后一格的聊天 UI
import gradio as gr
# OpenAI 官方客户端：chat.completions
from openai import OpenAI

# tiktoken：按模型估算知识库 token 数
import tiktoken
# numpy：嵌入向量转数组，供 t-SNE
import numpy as np
# OpenAIEmbeddings / ChatOpenAI：LangChain 封装的嵌入与聊天模型
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# Chroma：向量库
from langchain_chroma import Chroma
# HuggingFaceEmbeddings：备选本地嵌入（本练习主路径用 OpenAIEmbeddings）
from langchain_huggingface import HuggingFaceEmbeddings
# DirectoryLoader / TextLoader：批量读 knowledge-base 下 Markdown
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# RecursiveCharacterTextSplitter：字符级递归分块
from langchain_text_splitters import RecursiveCharacterTextSplitter
# TSNE：高维向量降到 2D/3D
from sklearn.manifold import TSNE
# plotly：交互散点图
import plotly.graph_objects as go

# SystemMessage / HumanMessage：给 ChatOpenAI.invoke 用的消息类型
from langchain_core.messages import SystemMessage, HumanMessage
# 课程附带的 evaluation.test（若同目录有 evaluation.py）
from evaluation import test

In [ ]:
# ========== 设置：加载密钥、选模型、建 OpenAI 客户端与 DB 名 ==========

# override=True：.env 覆盖已有同名变量
load_dotenv(override=True)
# 读取 OpenAI API Key
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    # 只打印前 8 位做「已加载」确认
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# 聊天 / 计 token 用的 model id（字符串保持原样）
MODEL = "gpt-4.1-nano"
# 默认从环境读密钥的 OpenAI 客户端
openai = OpenAI()

# Chroma 持久化目录名
DB_NAME = "vector_db"

In [ ]:
# ========== 打开已有 Chroma + 建 Retriever + system 模板 ==========

# 与建库时一致的 OpenAI 嵌入模型
embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")
# 打开（或创建空的）名为 docs 的 collection，目录为 DB_NAME
vector_db = Chroma(
    collection_name="docs",
    embedding_function=embedding_function,
    persist_directory=DB_NAME
)
# 每次检索取 top-5 相关块
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 嵌入式系统助手的 system 模板；{context} 运行时填入；正文不翻译
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant for embedded systems. Use the following context (MCUs, boards, sensors, tools, prices) to answer the question as accurately as possible.

Context:
{context}

If you do not know the answer, say you don't know. Do not make up information.
"""

In [ ]:
# ========== 用 LangChain ChatOpenAI 包装同一 MODEL ==========

# temperature=0：尽量稳定、少发散；model_name 与上面 MODEL 一致
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
# ========== RAG 回答函数：检索 → 填模板 → llm.invoke ==========

def answer_question(question: str, history):
    # 用问题做语义检索，得到 Document 列表
    docs = retriever.invoke(question)
    # 把各块正文用空行拼成一段 context
    context = "\n\n".join(doc.page_content for doc in docs)
    # 填入 system 模板
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    # SystemMessage + HumanMessage 一次调用
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    # 只返回文本内容
    return response.content

In [ ]:
# ========== 估算知识库总 token：优先用 chunks / documents，否则现加载 ==========

# 若前面已跑过分块或加载，优先复用内存中的对象
docs_for_text = globals().get("chunks") or globals().get("documents")
if docs_for_text is None:
    # 单元可单独运行：直接从 knowledge-base 加载全部 .md
    loader = DirectoryLoader("knowledge-base", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    docs_for_text = loader.load()
# 拼成一整段大文本
entire_knowledge_base = "\n\n".join(doc.page_content for doc in docs_for_text)

# 按 MODEL 选 tiktoken 编码器
encoding = tiktoken.encoding_for_model(MODEL)
# encode 后取长度 ≈ 输入 token 数
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# ========== 用 DirectoryLoader 加载 knowledge-base 下全部 Markdown ==========

# glob **/*.md：含子目录时也能扫到；仅扁平文件时同样可用
loader = DirectoryLoader("knowledge-base", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()
# 补 doc_type：没有则默认 "knowledge-base"（后面上色用）
for doc in documents:
    doc.metadata["doc_type"] = doc.metadata.get("doc_type", "knowledge-base")

print(f"Loaded {len(documents)} documents")

In [ ]:
# ========== RecursiveCharacterTextSplitter：1000 字符、重叠 200 ==========

# 与课程常见超参一致：块别太大也别丢边界上下文
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# 对刚加载的 documents 分块
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
# 打印第一块（含元数据的 Document 表示）
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
# ========== 选嵌入模型、清旧库、用 chunks 重建 Chroma ==========

# 与 implementation/answer.py 对齐，保证检索空间一致
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 若磁盘上已有同名库：删除 collection，避免新旧向量混杂
if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings, collection_name="docs").delete_collection()

# from_documents：算嵌入并持久化到 DB_NAME / collection docs
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
    collection_name="docs",
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# ========== 检查向量条数与单条嵌入维度 ==========

# 底层 collection 句柄
collection = vectorstore._collection
count = collection.count()

# 取 1 条嵌入，看维度（text-embedding-3-large 通常很高维）
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
# ========== 为可视化准备：向量矩阵、正文、类型颜色 ==========

# 一次性取出嵌入、文档正文、元数据
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
# 注意：这里 documents 被覆盖为「正文列表」，不再是 Document 对象
documents = result['documents']
metadatas = result['metadatas']
# 每条的 doc_type
doc_types = [metadata['doc_type'] for metadata in metadatas]
# 类型 → 颜色；未知回退 gray
type_colors = {'products': 'blue', 'employees': 'green', 'contracts': 'red', 'company': 'orange', 'knowledge-base': 'purple'}
colors = [type_colors.get(t, 'gray') for t in doc_types]

In [ ]:
# ========== 2D t-SNE：把高维向量压到平面，用 Plotly 散点查看 ==========

# 样本数决定 perplexity 上限（必须 < n_samples）
n_samples = vectors.shape[0]
perplexity = min(30, max(1, n_samples - 1))  # t-SNE requires perplexity < n_samples
# n_components=2：二维；random_state 固定可复现
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

# 二维散点：颜色按类型，hover 显示类型与正文前 100 字
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# ========== 3D t-SNE：再压一维，旋转观察簇结构 ==========

n_samples = vectors.shape[0]
perplexity = min(30, max(1, n_samples - 1))  # t-SNE requires perplexity < n_samples
# n_components=3 → Scatter3d
tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

# 三维散点图
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# ========== Gradio 聊天用的 system 前缀（后面再拼接检索上下文）==========

# 角色：嵌入式系统知识助手；正文影响行为，保持英文原样
SYSTEM_PREFIX = """
You represent an embedded systems knowledge assistant.
You are an expert in answering questions about embedded systems: MCUs, development boards, sensors, debuggers, RTOS, connectivity modules, and related components (prices, specs, part numbers).
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
# ========== 旧版关键字上下文（Gradio 路径实际走 additional_context / Retriever）==========

def get_relevant_context(message):
    # 只保留字母与空白，再按空格拆词
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    print(words)
    # 空字典：当前不会命中任何词；真正 RAG 在 additional_context
    knowledge = {}  # Not used when additional_context uses the vector-store retriever
    return [knowledge[word] for word in words if word in knowledge]   

In [ ]:
# ========== RAG 上下文：用 retriever 按用户消息取相关块 ==========

def additional_context(message):
    """从矢量存储中检索用户消息的相关嵌入式系统上下文。"""
    # 兼容 str 或带 .content 的对象
    content = message if isinstance(message, str) else (getattr(message, "content", "") or str(message))
    docs = retriever.invoke(content)
    if not docs:
        # 无可检索内容时的固定英文提示（保持原样）
        return "There is no additional context relevant to the user's question."
    # 多块正文用空行拼接
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
# ========== chat：system=前缀+检索上下文，再拼 history 与当前 user ==========

def chat(message, history):
    # SYSTEM_PREFIX + 检索到的 context → 完整 system
    system_message = SYSTEM_PREFIX + additional_context(message)
    # OpenAI messages：system + 历史 + 本轮 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 非流式一次生成
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
# ========== 启动 Gradio ChatInterface（messages 模式，自动开浏览器）==========

# type="messages"：history 为 role/content 字典列表，与上面 chat 拼接方式一致
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)